# Model Selection, Evaluation & Cross-Validation

## The Fundamental Question

You've cleaned your data. You've engineered features. You've built a model. It looks great on your data.

**But will it work on tomorrow's data?**

This is the core question of machine learning. A model that memorizes the past is useless. We need models that **generalize** to unseen data.

In this notebook, you'll learn:
- How to choose the right model for your problem
- How to measure model performance (and pick the right metric!)
- Why a single train/test split isn't enough
- **Cross-validation**: The gold standard for honest evaluation
- **TimeSeriesSplit**: The ONLY right way for financial data
- How to tune hyperparameters without overfitting

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline

# Model selection and evaluation
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    KFold, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
)

# Metrics
from sklearn.metrics import (
    # Regression
    mean_squared_error, mean_absolute_error, r2_score,
    # Classification
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Ready for model selection and evaluation!")

---

## Part 1: Model Selection - What Model Should I Use?

The first question: **What are you trying to predict?**

| Task | Target Type | Examples | Model Type |
|------|-------------|----------|------------|
| Regression | Continuous | Return, price, volatility | LinearRegression, Ridge, RandomForest |
| Classification | Discrete | Default/no-default, up/down | LogisticRegression, RandomForest, SVM |

### The Scikit-Learn Model Selection Cheat Sheet

Sklearn provides a helpful flowchart. Key decision points:
1. **How much data?** Some models need more data than others
2. **Predicting a category or quantity?** Classification vs Regression
3. **Need interpretability?** Linear models are easier to explain
4. **Working with text?** Use specialized text models

### Generate Example Data

Let's create two datasets: one for regression, one for classification.

In [ ]:
def generate_return_prediction_data(n_samples=1000, n_features=10, seed=42):
    """
    Generate data for return prediction (regression).
    Features are various factor exposures; target is next-period return.
    """
    np.random.seed(seed)
    
    # Generate features (think: factor exposures)
    X = np.random.randn(n_samples, n_features)
    
    # True coefficients (only first 5 features matter)
    true_coef = np.array([0.02, -0.015, 0.01, 0.008, -0.012] + [0]*(n_features-5))
    
    # Generate returns with realistic noise level
    signal = X @ true_coef
    noise = np.random.randn(n_samples) * 0.03  # ~3% noise std
    y = signal + noise
    
    feature_names = [f'factor_{i+1}' for i in range(n_features)]
    return pd.DataFrame(X, columns=feature_names), pd.Series(y, name='return')

def generate_default_prediction_data(n_samples=1000, seed=42):
    """
    Generate data for default prediction (classification).
    Features are borrower characteristics; target is default (1) or no-default (0).
    """
    np.random.seed(seed)
    
    # Features (standardized for easier coefficient interpretation)
    debt_to_income = np.random.exponential(0.4, n_samples)  # Higher mean
    credit_score = np.random.normal(680, 80, n_samples)     # Lower mean, higher variance
    years_employed = np.random.exponential(3, n_samples)    # Less experience
    loan_amount = np.random.lognormal(10, 0.8, n_samples)
    interest_rate = np.random.uniform(0.05, 0.20, n_samples)  # Higher rates
    
    # Create DataFrame
    X = pd.DataFrame({
        'debt_to_income': debt_to_income,
        'credit_score': credit_score,
        'years_employed': years_employed,
        'loan_amount': loan_amount,
        'interest_rate': interest_rate
    })
    
    # Generate default probability 
    # Coefficients tuned to give ~15-20% default rate
    log_odds = (
        -2 +  # Base rate
        2.0 * (debt_to_income - 0.4) +           # Higher debt = more defaults
        -0.03 * (credit_score - 680) +           # Lower credit = more defaults
        -0.15 * (years_employed - 3) +           # Less experience = more defaults
        0.00001 * (loan_amount - 30000) +        # Larger loans = slightly more risk
        8 * (interest_rate - 0.125)              # Higher rate = more defaults
    )
    prob_default = 1 / (1 + np.exp(-log_odds))
    y = (np.random.random(n_samples) < prob_default).astype(int)
    
    return X, pd.Series(y, name='default')

# Generate datasets
X_reg, y_reg = generate_return_prediction_data()
X_clf, y_clf = generate_default_prediction_data()

print("Regression Dataset (Return Prediction):")
print(f"  Samples: {len(X_reg)}, Features: {X_reg.shape[1]}")
print(f"  Target mean: {y_reg.mean():.4f}, std: {y_reg.std():.4f}")

print("\nClassification Dataset (Default Prediction):")
print(f"  Samples: {len(X_clf)}, Features: {X_clf.shape[1]}")
print(f"  Default rate: {y_clf.mean():.1%}")
print(f"  Class distribution: {y_clf.value_counts().to_dict()}")

### Quick Model Comparison

In [ ]:
# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Compare several regression models
regression_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Lasso (alpha=0.001)': Lasso(alpha=0.001),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
}

print("Regression Model Comparison:")
print("-" * 50)
for name, model in regression_models.items():
    pipe = make_pipeline(StandardScaler(), model)
    pipe.fit(X_train_reg, y_train_reg)
    y_pred = pipe.predict(X_test_reg)
    mse = mean_squared_error(y_test_reg, y_pred)
    r2 = r2_score(y_test_reg, y_pred)
    print(f"{name:25s} | MSE: {mse:.6f} | R²: {r2:.4f}")

In [ ]:
# Split data
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf  # stratify keeps class balance
)

# Compare several classification models
classification_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
}

print("Classification Model Comparison:")
print("-" * 60)
for name, model in classification_models.items():
    pipe = make_pipeline(StandardScaler(), model)
    pipe.fit(X_train_clf, y_train_clf)
    y_pred = pipe.predict(X_test_clf)
    y_prob = pipe.predict_proba(X_test_clf)[:, 1]
    acc = accuracy_score(y_test_clf, y_pred)
    auc = roc_auc_score(y_test_clf, y_prob)
    print(f"{name:25s} | Accuracy: {acc:.4f} | AUC: {auc:.4f}")

---

## Part 2: Evaluation Metrics - How Good Is My Model?

**The metric you choose shapes everything.** Different metrics reward different behaviors.

### 2.1 Regression Metrics

In [ ]:
# Train a simple model for demonstration
model_reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model_reg.fit(X_train_reg, y_train_reg)
y_pred_reg = model_reg.predict(X_test_reg)

In [ ]:
# Mean Squared Error (MSE)
mse = mean_squared_error(y_test_reg, y_pred_reg)
print(f"MSE: {mse:.6f}")
print("  - Penalizes large errors more (squared)")
print("  - Units: (target units)²")
print("  - Lower is better")

In [ ]:
# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.6f}")
print("  - Same units as target (easier to interpret)")
print(f"  - 'On average, predictions are off by {rmse:.4f}'")

In [ ]:
# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
print(f"MAE: {mae:.6f}")
print("  - Treats all errors equally (no squaring)")
print("  - More robust to outliers than MSE")
print("  - Same units as target")

In [ ]:
# R² Score (Coefficient of Determination)
r2 = r2_score(y_test_reg, y_pred_reg)
print(f"R²: {r2:.4f}")
print("  - Proportion of variance explained by the model")
print("  - 1.0 = perfect, 0.0 = no better than predicting the mean")
print("  - Can be negative if model is worse than mean!")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.5, edgecolors='none')
axes[0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Return')
axes[0].set_ylabel('Predicted Return')
axes[0].set_title(f'Actual vs Predicted (R² = {r2:.4f})')
axes[0].legend()

# Error distribution
errors = y_test_reg - y_pred_reg
axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Error Distribution (MAE = {mae:.4f})')

plt.tight_layout()

### Which Regression Metric to Use?

| Metric | Use When | Watch Out For |
|--------|----------|---------------|
| **MSE/RMSE** | Large errors are especially bad | Sensitive to outliers |
| **MAE** | All errors equally important | Doesn't penalize big errors much |
| **R²** | Comparing models on same dataset | Can be misleading across datasets |
| **MAPE** | Want percentage errors | Undefined if actuals are zero |

### 2.2 Classification Metrics

Classification has more nuance because errors aren't symmetric. Missing a default (False Negative) vs. flagging a good loan (False Positive) have different costs.

In [ ]:
# Train a classifier
model_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_clf.fit(X_train_clf, y_train_clf)
y_pred_clf = model_clf.predict(X_test_clf)
y_prob_clf = model_clf.predict_proba(X_test_clf)[:, 1]

In [ ]:
# Confusion Matrix - The Foundation
cm = confusion_matrix(y_test_clf, y_pred_clf)
print("Confusion Matrix:")
print(f"                 Predicted")
print(f"              No-Default  Default")
print(f"Actual No-Default  {cm[0,0]:4d}      {cm[0,1]:4d}")
print(f"       Default     {cm[1,0]:4d}      {cm[1,1]:4d}")
print("\nInterpretation:")
print(f"  True Negatives (TN): {cm[0,0]} - Correctly predicted no-default")
print(f"  False Positives (FP): {cm[0,1]} - Incorrectly predicted default")
print(f"  False Negatives (FN): {cm[1,0]} - Missed actual defaults!")
print(f"  True Positives (TP): {cm[1,1]} - Correctly caught defaults")

In [ ]:
# Accuracy
acc = accuracy_score(y_test_clf, y_pred_clf)
print(f"Accuracy: {acc:.4f}")
print(f"  - {acc:.1%} of predictions were correct")
print("  - WARNING: Can be misleading with imbalanced classes!")
print(f"  - If we predicted 'no-default' for everyone: {1 - y_test_clf.mean():.1%} accuracy")

In [ ]:
# Precision and Recall
precision = precision_score(y_test_clf, y_pred_clf)
recall = recall_score(y_test_clf, y_pred_clf)

print(f"Precision: {precision:.4f}")
print(f"  - When we predict default, we're right {precision:.1%} of the time")
print(f"  - 'How many predicted positives are actually positive?'")

print(f"\nRecall: {recall:.4f}")
print(f"  - We catch {recall:.1%} of actual defaults")
print(f"  - 'How many actual positives did we find?'")

In [ ]:
# F1 Score - Harmonic mean of precision and recall
f1 = f1_score(y_test_clf, y_pred_clf)
print(f"F1 Score: {f1:.4f}")
print(f"  - Balances precision and recall")
print(f"  - Useful when you want both to be good")

In [ ]:
# AUC-ROC - The Best Single Metric for Classification
auc = roc_auc_score(y_test_clf, y_prob_clf)
print(f"AUC-ROC: {auc:.4f}")
print(f"  - Area Under the ROC Curve")
print(f"  - Probability that model ranks a random positive higher than a random negative")
print(f"  - 0.5 = random guessing, 1.0 = perfect")
print(f"  - Threshold-independent!")

In [ ]:
# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test_clf, y_prob_clf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, linewidth=2, label=f'Logistic Regression (AUC = {auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Guess (AUC = 0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Probability distribution by class
axes[1].hist(y_prob_clf[y_test_clf == 0], bins=30, alpha=0.5, label='No Default', density=True)
axes[1].hist(y_prob_clf[y_test_clf == 1], bins=30, alpha=0.5, label='Default', density=True)
axes[1].axvline(x=0.5, color='r', linestyle='--', label='Threshold = 0.5')
axes[1].set_xlabel('Predicted Probability of Default')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by Class')
axes[1].legend()

plt.tight_layout()

In [ ]:
# Complete classification report
print("Complete Classification Report:")
print("=" * 50)
print(classification_report(y_test_clf, y_pred_clf, target_names=['No Default', 'Default']))

### Which Classification Metric to Use?

| Metric | Use When | Example |
|--------|----------|----------|
| **Accuracy** | Classes are balanced, all errors equal | Spam detection (balanced) |
| **Precision** | False positives are costly | Flagging for manual review |
| **Recall** | False negatives are costly | Fraud detection, disease screening |
| **F1** | Need balance of precision/recall | Most imbalanced problems |
| **AUC-ROC** | Ranking quality matters | Credit scoring, default prediction |

---

## Part 3: Why Cross-Validation?

A single train/test split has a problem: **the result depends on which samples ended up in test**.

Let's demonstrate:

In [ ]:
# Run 10 different train/test splits
r2_scores = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=seed)
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(X_tr, y_tr)
    r2_scores.append(r2_score(y_te, model.predict(X_te)))

print("R² scores across 10 different random splits:")
print([f"{s:.4f}" for s in r2_scores])
print(f"\nMean: {np.mean(r2_scores):.4f}")
print(f"Std:  {np.std(r2_scores):.4f}")
print(f"Range: {np.min(r2_scores):.4f} to {np.max(r2_scores):.4f}")
print(f"\nThe same model can look {(np.max(r2_scores) - np.min(r2_scores))/np.mean(r2_scores)*100:.0f}% better or worse depending on the split!")

### K-Fold Cross-Validation

**Solution**: Use ALL data for both training and testing, but never at the same time.

1. Split data into K equal parts ("folds")
2. Train on K-1 folds, test on the remaining fold
3. Repeat K times, using each fold as test once
4. Average the results

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

In [ ]:
# K-Fold Cross-Validation
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

# cross_val_score does all the work!
cv_scores = cross_val_score(model, X_reg, y_reg, cv=5, scoring='r2')

print("5-Fold Cross-Validation:")
print(f"R² scores: {cv_scores.round(4)}")
print(f"Mean R²: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
print("\nNow we have a confidence interval, not just a single number!")

In [ ]:
# Visualize the folds
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fig, ax = plt.subplots(figsize=(12, 4))

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X_reg)):
    # Plot training indices
    ax.scatter(train_idx, [fold_idx]*len(train_idx), c='blue', s=1, alpha=0.5)
    # Plot test indices
    ax.scatter(test_idx, [fold_idx]*len(test_idx), c='red', s=5)

ax.set_xlabel('Sample Index')
ax.set_ylabel('Fold')
ax.set_title('K-Fold Cross-Validation Splits (blue=train, red=test)')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
plt.tight_layout()

---

## Part 4: TimeSeriesSplit - The Right Way for Financial Data

**Standard K-Fold is WRONG for time series!**

Why? Because it allows future data to help predict the past. This is **lookahead bias**.

In the diagram above, notice how test samples (red) are scattered throughout the data. In time series, this means:
- Fold 1 tests on sample 200, but trains on sample 800 (future!)
- This creates unrealistic performance estimates

In [ ]:
# Generate time series data
def generate_time_series_returns(n_days=500, seed=42):
    """Generate time series with some predictability."""
    np.random.seed(seed)
    
    dates = pd.date_range(end='2024-12-31', periods=n_days, freq='B')
    
    # Features: lagged returns, volatility
    returns = np.random.randn(n_days) * 0.02  # Daily returns
    
    # Add some mean reversion (makes it slightly predictable)
    for i in range(5, n_days):
        returns[i] -= 0.1 * np.mean(returns[i-5:i])  # Mean reversion
        returns[i] += np.random.randn() * 0.02
    
    # Create features
    df = pd.DataFrame({'return': returns}, index=dates)
    df['lag1'] = df['return'].shift(1)
    df['lag5'] = df['return'].shift(5)
    df['vol5'] = df['return'].rolling(5).std()
    df['vol20'] = df['return'].rolling(20).std()
    df['momentum20'] = df['return'].rolling(20).mean()
    
    # Target: next day return
    df['target'] = df['return'].shift(-1)
    
    df = df.dropna()
    
    features = ['lag1', 'lag5', 'vol5', 'vol20', 'momentum20']
    X = df[features]
    y = df['target']
    
    return X, y

X_ts, y_ts = generate_time_series_returns()
print(f"Time series data: {len(X_ts)} days")
print(f"Date range: {X_ts.index[0].date()} to {X_ts.index[-1].date()}")
X_ts.head()

In [ ]:
# WRONG: Standard K-Fold on time series
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

kfold_scores = cross_val_score(model, X_ts, y_ts, cv=5, scoring='r2')

print("WRONG - Standard K-Fold on Time Series:")
print(f"R² scores: {kfold_scores.round(4)}")
print(f"Mean R²: {kfold_scores.mean():.4f}")
print("\nThis is BIASED - it uses future data to predict the past!")

In [ ]:
# RIGHT: TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

tscv_scores = cross_val_score(model, X_ts, y_ts, cv=tscv, scoring='r2')

print("RIGHT - TimeSeriesSplit:")
print(f"R² scores: {tscv_scores.round(4)}")
print(f"Mean R²: {tscv_scores.mean():.4f}")
print("\nThis is HONEST - it only uses past data to predict future!")

In [ ]:
# Visualize TimeSeriesSplit
fig, ax = plt.subplots(figsize=(14, 4))

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_ts)):
    ax.scatter(train_idx, [fold_idx]*len(train_idx), c='blue', s=1, alpha=0.5)
    ax.scatter(test_idx, [fold_idx]*len(test_idx), c='red', s=5)

ax.set_xlabel('Sample Index (Time →)')
ax.set_ylabel('Fold')
ax.set_title('TimeSeriesSplit (blue=train, red=test) - Test is ALWAYS after train!')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
plt.tight_layout()

### Key Observation

Notice how TimeSeriesSplit:
- Always tests on data that comes AFTER the training data
- Training set grows over time (expanding window)
- This mimics real-world model deployment!

Compare the R² scores:
- K-Fold (with leakage): Higher scores (biased)
- TimeSeriesSplit (honest): Lower scores (realistic)

In [ ]:
# Compare K-Fold vs TimeSeriesSplit
print("Comparison:")
print(f"K-Fold Mean R²:       {kfold_scores.mean():.4f}")
print(f"TimeSeriesSplit R²:   {tscv_scores.mean():.4f}")
print(f"\nDifference:          {(kfold_scores.mean() - tscv_scores.mean()):.4f}")
print("\nK-Fold gives OVERLY OPTIMISTIC results for time series data!")
print("Always use TimeSeriesSplit for financial data.")

---

## Part 5: Hyperparameter Tuning

Most models have **hyperparameters** - settings you choose *before* training that affect model behavior:

| Model | Hyperparameters | What They Control |
|-------|-----------------|-------------------|
| **Ridge Regression** | `alpha` | Regularization strength (how much to penalize large coefficients) |
| **Random Forest** | `n_estimators`, `max_depth`, `min_samples_split` | Number of trees, tree complexity, when to stop splitting |
| **LLMs (GPT, Claude)** | `temperature`, `top_p`, `max_tokens` | Randomness/creativity, nucleus sampling, response length |

**Key insight**: Hyperparameter tuning isn't just for traditional ML! When you adjust `temperature` in ChatGPT or Claude, you're tuning a hyperparameter. The difference is:
- **Sklearn models**: We tune hyperparameters to maximize predictive accuracy (MSE, AUC, etc.)
- **LLMs**: We tune hyperparameters to control output style (more creative vs more deterministic)


For sklearn models, we use **GridSearchCV** or **RandomizedSearchCV** to systematically find the best hyperparameters.

### 5.1 GridSearchCV: Exhaustive Search

In [ ]:
# Define the pipeline
pipeline = make_pipeline(StandardScaler(), Ridge())

# Define hyperparameter grid
# Note: use 'ridge__alpha' because it's step 'ridge' in pipeline
param_grid = {
    'ridge__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# Use TimeSeriesSplit for honest evaluation
tscv = TimeSeriesSplit(n_splits=5)

# GridSearchCV
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=tscv, 
    scoring='neg_mean_squared_error',  # sklearn uses negative (higher is better)
    return_train_score=True
)

grid_search.fit(X_ts, y_ts)

print("GridSearchCV Results:")
print(f"Best alpha: {grid_search.best_params_['ridge__alpha']}")
print(f"Best CV MSE: {-grid_search.best_score_:.6f}")

In [ ]:
# Visualize results
results = pd.DataFrame(grid_search.cv_results_)

fig, ax = plt.subplots(figsize=(10, 6))

alphas = results['param_ridge__alpha'].astype(float)
train_scores = -results['mean_train_score']
test_scores = -results['mean_test_score']

ax.plot(alphas, train_scores, 'o-', label='Train MSE', linewidth=2)
ax.plot(alphas, test_scores, 's-', label='CV Test MSE', linewidth=2)
ax.axvline(x=grid_search.best_params_['ridge__alpha'], color='r', linestyle='--', 
           label=f'Best alpha = {grid_search.best_params_["ridge__alpha"]}')

ax.set_xscale('log')
ax.set_xlabel('Alpha (regularization strength)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Hyperparameter Tuning: Ridge Regression')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

### 5.2 RandomizedSearchCV: When Grid is Too Large

Random Forest has many hyperparameters. GridSearch becomes infeasible.

In [ ]:
from scipy.stats import randint, uniform

# Random Forest with randomized search
pipeline_rf = make_pipeline(StandardScaler(), RandomForestRegressor(random_state=42))

# Define distributions for random sampling
param_distributions = {
    'randomforestregressor__n_estimators': randint(50, 200),
    'randomforestregressor__max_depth': [5, 10, 20, None],
    'randomforestregressor__min_samples_split': randint(2, 20),
    'randomforestregressor__min_samples_leaf': randint(1, 10),
}

# RandomizedSearchCV - try 20 random combinations
random_search = RandomizedSearchCV(
    pipeline_rf,
    param_distributions,
    n_iter=20,  # Number of random combinations to try
    cv=tscv,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

random_search.fit(X_ts, y_ts)

print("RandomizedSearchCV Results:")
print(f"Best parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest CV MSE: {-random_search.best_score_:.6f}")

---

## Part 6: Putting It All Together

Let's build a complete ML pipeline from start to finish, following best practices.

In [ ]:
# Generate fresh time series data
X_full, y_full = generate_time_series_returns(n_days=750, seed=123)
print(f"Full dataset: {len(X_full)} samples")

# Hold out the last 20% as a TRUE test set (never touched during CV)
split_point = int(len(X_full) * 0.8)
X_dev, y_dev = X_full.iloc[:split_point], y_full.iloc[:split_point]
X_final_test, y_final_test = X_full.iloc[split_point:], y_full.iloc[split_point:]

print(f"Development set (for CV): {len(X_dev)} samples")
print(f"Final test set (held out): {len(X_final_test)} samples")

In [ ]:
# Step 1: Define candidate models
candidate_models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso': make_pipeline(StandardScaler(), Lasso(alpha=0.001)),
    'Random Forest': make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42)),
}

# Step 2: Compare using TimeSeriesSplit CV
tscv = TimeSeriesSplit(n_splits=5)

print("Step 1: Model Selection via Cross-Validation")
print("=" * 60)
cv_results = {}
for name, model in candidate_models.items():
    scores = cross_val_score(model, X_dev, y_dev, cv=tscv, scoring='neg_mean_squared_error')
    mse_scores = -scores
    cv_results[name] = mse_scores
    print(f"{name:20s} | CV MSE: {mse_scores.mean():.6f} (+/- {mse_scores.std()*2:.6f})")

best_model_name = min(cv_results, key=lambda k: cv_results[k].mean())
print(f"\nBest model: {best_model_name}")

In [ ]:
# Step 3: Hyperparameter tuning for best model
print("\nStep 2: Hyperparameter Tuning for Ridge")
print("=" * 60)

pipeline = make_pipeline(StandardScaler(), Ridge())
param_grid = {'ridge__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

grid_search = GridSearchCV(pipeline, param_grid, cv=tscv, scoring='neg_mean_squared_error')
grid_search.fit(X_dev, y_dev)

print(f"Best alpha: {grid_search.best_params_['ridge__alpha']}")
print(f"Best CV MSE: {-grid_search.best_score_:.6f}")

In [ ]:
# Step 4: Final evaluation on held-out test set
print("\nStep 3: Final Evaluation on Held-Out Test Set")
print("=" * 60)

# Train best model on ALL development data
final_model = grid_search.best_estimator_
final_model.fit(X_dev, y_dev)

# Predict on final test set
y_pred_final = final_model.predict(X_final_test)

# Calculate metrics
final_mse = mean_squared_error(y_final_test, y_pred_final)
final_mae = mean_absolute_error(y_final_test, y_pred_final)
final_r2 = r2_score(y_final_test, y_pred_final)

print(f"Final Test Set Results:")
print(f"  MSE:  {final_mse:.6f}")
print(f"  RMSE: {np.sqrt(final_mse):.6f}")
print(f"  MAE:  {final_mae:.6f}")
print(f"  R²:   {final_r2:.4f}")

In [ ]:
# Visualize final results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series of predictions vs actual
axes[0].plot(y_final_test.index, y_final_test.values, label='Actual', alpha=0.7)
axes[0].plot(y_final_test.index, y_pred_final, label='Predicted', alpha=0.7)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Return')
axes[0].set_title('Final Test Set: Actual vs Predicted Returns')
axes[0].legend()

# Cumulative returns comparison
cumret_actual = (1 + y_final_test).cumprod()
cumret_pred_signal = (1 + y_final_test * np.sign(y_pred_final)).cumprod()  # Simple signal strategy

axes[1].plot(cumret_actual.index, cumret_actual.values, label='Buy & Hold')
axes[1].plot(cumret_actual.index, cumret_pred_signal.values, label='Model Signal')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Strategy Comparison (if you traded on model signal)')
axes[1].legend()

plt.tight_layout()

---

## Summary: Key Takeaways

### Model Selection
```python
# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
```

### Evaluation Metrics
```python
from sklearn.metrics import mean_squared_error, r2_score  # Regression
from sklearn.metrics import accuracy_score, roc_auc_score  # Classification
```

### Cross-Validation
```python
from sklearn.model_selection import cross_val_score, TimeSeriesSplit

# Standard CV (NOT for time series)
scores = cross_val_score(model, X, y, cv=5)

# Time series CV (USE THIS for financial data)
tscv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(model, X, y, cv=tscv)
```

### Hyperparameter Tuning
```python
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# GridSearchCV (exhaustive)
grid_search = GridSearchCV(model, param_grid, cv=tscv)
grid_search.fit(X, y)
best_model = grid_search.best_estimator_

# RandomizedSearchCV (faster for large grids)
random_search = RandomizedSearchCV(model, param_distributions, n_iter=20, cv=tscv)
```

### The Complete Workflow
1. **Split data**: Development set + Final test set
2. **Select model**: Compare candidates using CV on development set
3. **Tune hyperparameters**: GridSearchCV on development set
4. **Final evaluation**: Train on all development data, test on final test set
5. **Report honestly**: Use final test set metrics (not CV metrics)

### Critical Rules for Finance

1. **Always use TimeSeriesSplit** - Never allow future data to help predict the past
2. **Hold out a final test set** - CV scores are still slightly optimistic from hyperparameter tuning
3. **Report realistic metrics** - Don't cherry-pick the best fold
4. **Be skeptical of high R²** - In finance, R² of 0.01 can be economically significant
5. **Consider transaction costs** - A model that predicts well might not be profitable

---

## Congratulations!

You've completed Lecture 7. You now understand:
- How sklearn is designed and why it works
- How to build feature engineering pipelines
- How to choose and evaluate models
- **Most importantly**: How to do this honestly for financial data

These skills are fundamental for any quantitative role. Whether you're building alpha models, risk systems, or pricing engines, you'll use these techniques daily.

**Next steps**:
- Practice on real financial datasets (Yahoo Finance, FRED, Quandl)
- Explore more advanced topics: ensemble methods, neural networks, feature selection
- Read "Advances in Financial Machine Learning" by Marcos López de Prado

Good luck on your quant journey!